Importing Libaries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Loading the datasets


In [1]:
import pandas as pd

files = ["dataset/Components of population change - states and territories.xlsx",
         "dataset/Population - New South Wales.xlsx",
         "dataset/Population - states and territories.xlsx"]
dataframes = []

for file in files:
    try:
        df = pd.read_excel(file, sheet_name="Data1")

        # remove top metadata rows
        df = df.iloc[9:].reset_index(drop=True)

        # create new dataframe
        new_df = pd.DataFrame()
        new_df["date"] = pd.to_datetime(df.iloc[:, 0], errors="coerce")

        # add extra columns
        new_df["year"] = new_df["date"].dt.year
        new_df["quarter"] = new_df["date"].dt.quarter
        new_df["source_file"] = file

        # add all other columns
        for i, col in enumerate(df.columns[1:]):
            name = str(col).split(";")
            measure = name[0].strip()
            state = name[1].strip() if len(name) > 1 else "unknown"

            col_name = f"{measure}_{state}".lower().replace(" ", "_")
            new_df[col_name] = pd.to_numeric(df.iloc[:, i+1], errors="coerce")

        dataframes.append(new_df)

    except Exception as e:
        print("Error:", file, e)

COMBINE DATA

In [7]:
print("\nCombining data from all files...")

final_df = pd.concat(dataframes, ignore_index=True)
final_df = final_df.dropna(subset=['date', 'year'])

print(f"Combined data shape: {final_df.shape}")
print(f"\nDate range: {final_df['date'].min()} to {final_df['date'].max()}")


Combining data from all files...
Combined data shape: (409, 42)

Date range: 1971-06-01 00:00:00 to 2025-06-01 00:00:00


Data Preprocessing

In [3]:
print("\n[STAGE 2] Data Preprocessing")

df_analysis = model_data.copy()

# Checking for missing values
print(f"\nMissing values check:")
print(df_analysis.isnull().sum())

# Log transformations
df_analysis['NI_Log'] = np.log(df_analysis['natural_increase_au'] + 1)
df_analysis['NM_Log'] = np.log(df_analysis['net_overseas_migration_au'] + 1)

# Detect and handle outliers using IQR method
print(f"\nOutlier detection using IQR method:")
outliers_detected = 0

for col in ['natural_increase_au', 'net_overseas_migration_au']:
    Q1 = df_analysis[col].quantile(0.25)
    Q3 = df_analysis[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5*IQR
    upper_bound = Q3 + 1.5*IQR
    
    outliers = (df_analysis[col] < lower_bound) | (df_analysis[col] > upper_bound)
    n_outliers = outliers.sum()
    
    if n_outliers > 0:
        print(f"  {col}: {n_outliers} outliers detected")
        outliers_detected += n_outliers
        df_analysis.loc[outliers, col] = df_analysis[col].median()

print(f"\nTotal outliers corrected: {outliers_detected}")
print(f"Preprocessed shape: {df_analysis.shape}")
print(f"Ready for analysis")



[STAGE 2] Data Preprocessing


NameError: name 'model_data' is not defined

Feature Engineering

In [2]:
print("\n[STAGE 4] Feature Engineering")

# Creating the lagged variables which will help find the differences in quaters


df_analysis['NI_Lag1'] = df_analysis['natural_increase_au'].shift(1)
df_analysis['NI_Lag2'] = df_analysis['natural_increase_au'].shift(2)
df_analysis['NM_Lag1'] = df_analysis['net_overseas_migration_au'].shift(1)
df_analysis['NM_Lag2'] = df_analysis['net_overseas_migration_au'].shift(2)

# Creating volatility measures, this changes over time
df_analysis['NI_Volatility'] = df_analysis['natural_increase_au'].rolling(window=4).std()
df_analysis['NM_Volatility'] = df_analysis['net_overseas_migration_au'].rolling(window=4).std()

# Creating momentum indicators
df_analysis['NI_Momentum'] = df_analysis['natural_increase_au'].diff()
df_analysis['NM_Momentum'] = df_analysis['net_overseas_migration_au'].diff()

# This shows interaction between NI and NM
df_analysis['NI_x_NM'] = df_analysis['natural_increase_au'] * df_analysis['net_overseas_migration_au']

# Remove rows with NaN values created by lags and rolling windows
df_model = df_analysis.dropna()

print(f"Features created:")
print(f"  - Lagged variables (1 quarter)")
print(f"  - Volatility measures (4-quarter rolling std)")
print(f"  - Momentum indicators")
print(f"  - Interaction terms")
print(f"\nFinal modeling sample: {len(df_model)} observations")


[STAGE 4] Feature Engineering


NameError: name 'df_analysis' is not defined